In [ ]:
!pip install apache-airflow


In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 19.8 MB/s eta 0:00:00


In [ ]:
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import avg
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [ ]:
db = sqlite3.connect("Cadd_pipeline")

In [ ]:
data = pd.read_csv("/content/clean_data.csv")

In [ ]:
data.to_sql("molecules",db,index=False,if_exists="replace")

16067

In [ ]:
result = pd.read_sql("select * from molecules limit 5 ",db)
result

,SMILES,pIC50,mol,num_atoms,logP
0,O=S(=O)(Nc1cccc(-c2cnc3ccccc3n2)c1)c1cccs1,4.26,<rdkit.Chem.rdchem.Mol object at 0x7f59df45bc30>,25,4.1591
1,O=c1cc(-c2nc(-c3ccc(-c4cn(CCP(=O)(O)O)nn4)cc3)...,4.34,<rdkit.Chem.rdchem.Mol object at 0x7f59a320c9e0>,36,3.6743
2,NC(=O)c1ccc2c(c1)nc(C1CCC(O)CC1)n2CCCO,4.53,<rdkit.Chem.rdchem.Mol object at 0x7f59a320cac0>,23,1.5361
3,NCCCn1c(C2CCNCC2)nc2cc(C(N)=O)ccc21,4.56,<rdkit.Chem.rdchem.Mol object at 0x7f59a320cba0>,22,0.9510
4,CNC(=S)Nc1cccc(-c2cnc3ccccc3n2)c1,4.59,<rdkit.Chem.rdchem.Mol object at 0x7f59a320c7b0>,21,3.2130


In [ ]:
result1=pd.read_sql("select * from molecules where num_atoms < 60 limit 5",db)
result1


,SMILES,pIC50,mol,num_atoms,logP
0,O=S(=O)(Nc1cccc(-c2cnc3ccccc3n2)c1)c1cccs1,4.26,<rdkit.Chem.rdchem.Mol object at 0x7f59df45bc30>,25,4.1591
1,O=c1cc(-c2nc(-c3ccc(-c4cn(CCP(=O)(O)O)nn4)cc3)...,4.34,<rdkit.Chem.rdchem.Mol object at 0x7f59a320c9e0>,36,3.6743
2,NC(=O)c1ccc2c(c1)nc(C1CCC(O)CC1)n2CCCO,4.53,<rdkit.Chem.rdchem.Mol object at 0x7f59a320cac0>,23,1.5361
3,NCCCn1c(C2CCNCC2)nc2cc(C(N)=O)ccc21,4.56,<rdkit.Chem.rdchem.Mol object at 0x7f59a320cba0>,22,0.9510
4,CNC(=S)Nc1cccc(-c2cnc3ccccc3n2)c1,4.59,<rdkit.Chem.rdchem.Mol object at 0x7f59a320c7b0>,21,3.2130


In [ ]:
result2=pd.read_sql("select count(*) from molecules",db)
result2

,count(*)
0,16067


In [ ]:
spark = SparkSession.builder.appName("Cadd").getOrCreate()

In [ ]:
spark_df = spark.read.csv("clean_data.csv",header=True,inferSchema=True)

In [ ]:
spark_df.show(10)

+--------------------+-----+--------------------+---------+------------------+
|              SMILES|pIC50|                 mol|num_atoms|              logP|
+--------------------+-----+--------------------+---------+------------------+
|O=S(=O)(Nc1cccc(-...| 4.26|<rdkit.Chem.rdche...|       25| 4.159100000000002|
|O=c1cc(-c2nc(-c3c...| 4.34|<rdkit.Chem.rdche...|       36|3.6743000000000015|
|NC(=O)c1ccc2c(c1)...| 4.53|<rdkit.Chem.rdche...|       23|            1.5361|
|NCCCn1c(C2CCNCC2)...| 4.56|<rdkit.Chem.rdche...|       22|0.9509999999999998|
|CNC(=S)Nc1cccc(-c...| 4.59|<rdkit.Chem.rdche...|       21| 3.213000000000001|
|O=C(Nc1cccc(-c2cn...|  4.6|<rdkit.Chem.rdche...|       22|3.6453000000000015|
|O=C(Nc1cccc(-c2cn...| 4.61|<rdkit.Chem.rdche...|       24| 4.142100000000002|
|COc1cc(Nc2nccc(-c...| 4.67|<rdkit.Chem.rdche...|       26| 4.566400000000004|
|CN1C(=O)COc2c1cnc...| 4.68|<rdkit.Chem.rdche...|       33| 3.323500000000002|
|COc1ccc2c(c1)C(=O...| 4.68|<rdkit.Chem.rdche...|   

In [ ]:
spark_df.filter("num_atoms < 60").show(10)

+--------------------+-----+--------------------+---------+------------------+
|              SMILES|pIC50|                 mol|num_atoms|              logP|
+--------------------+-----+--------------------+---------+------------------+
|O=S(=O)(Nc1cccc(-...| 4.26|<rdkit.Chem.rdche...|       25| 4.159100000000002|
|O=c1cc(-c2nc(-c3c...| 4.34|<rdkit.Chem.rdche...|       36|3.6743000000000015|
|NC(=O)c1ccc2c(c1)...| 4.53|<rdkit.Chem.rdche...|       23|            1.5361|
|NCCCn1c(C2CCNCC2)...| 4.56|<rdkit.Chem.rdche...|       22|0.9509999999999998|
|CNC(=S)Nc1cccc(-c...| 4.59|<rdkit.Chem.rdche...|       21| 3.213000000000001|
|O=C(Nc1cccc(-c2cn...|  4.6|<rdkit.Chem.rdche...|       22|3.6453000000000015|
|O=C(Nc1cccc(-c2cn...| 4.61|<rdkit.Chem.rdche...|       24| 4.142100000000002|
|COc1cc(Nc2nccc(-c...| 4.67|<rdkit.Chem.rdche...|       26| 4.566400000000004|
|CN1C(=O)COc2c1cnc...| 4.68|<rdkit.Chem.rdche...|       33| 3.323500000000002|
|COc1ccc2c(c1)C(=O...| 4.68|<rdkit.Chem.rdche...|   

In [ ]:
spark_df.select("SMILES","pIC50","num_atoms").show(10)

+--------------------+-----+---------+
|              SMILES|pIC50|num_atoms|
+--------------------+-----+---------+
|O=S(=O)(Nc1cccc(-...| 4.26|       25|
|O=c1cc(-c2nc(-c3c...| 4.34|       36|
|NC(=O)c1ccc2c(c1)...| 4.53|       23|
|NCCCn1c(C2CCNCC2)...| 4.56|       22|
|CNC(=S)Nc1cccc(-c...| 4.59|       21|
|O=C(Nc1cccc(-c2cn...|  4.6|       22|
|O=C(Nc1cccc(-c2cn...| 4.61|       24|
|COc1cc(Nc2nccc(-c...| 4.67|       26|
|CN1C(=O)COc2c1cnc...| 4.68|       33|
|COc1ccc2c(c1)C(=O...| 4.68|       32|
+--------------------+-----+---------+
only showing top 10 rows


In [ ]:
spark_df = spark_df.withColumn(
    "double_atoms",
    col("num_atoms")*2
)

In [ ]:
spark_df.groupBy("num_atoms").count().show(10)

+---------+-----+
|num_atoms|count|
+---------+-----+
|       31|  202|
|       53|    9|
|       34|  173|
|       28|  298|
|       26|  356|
|       27|  334|
|       44|   21|
|       12|  858|
|       22|  534|
|       47|   10|
+---------+-----+
only showing top 10 rows


In [ ]:
spark_df.select(avg("pIC50")).show(10)

+------------------+
|        avg(pIC50)|
+------------------+
|0.9999480964527225|
+------------------+



In [ ]:
spark_df.orderBy(col("pIC50").desc()).show(10)

+--------------------+-----+--------------------+---------+------------------+------------+
|              SMILES|pIC50|                 mol|num_atoms|              logP|double_atoms|
+--------------------+-----+--------------------+---------+------------------+------------+
|CS(=O)(=O)N1CCC(N...|10.97|<rdkit.Chem.rdche...|       25|1.4656999999999996|          50|
|COC(=O)N1CCCCC(Nc...|10.97|<rdkit.Chem.rdche...|       27|3.0527000000000006|          54|
|CC1CN(S(=O)(=O)CC...|10.78|<rdkit.Chem.rdche...|       29|1.7944999999999998|          58|
|CN1CCN(c2ncc(-c3c...|10.76|<rdkit.Chem.rdche...|       38|             2.809|          76|
|CC(=O)N1CCCCC(Nc2...|10.72|<rdkit.Chem.rdche...|       26|2.8328000000000007|          52|
|CC(C)CS(=O)(=O)N1...|10.71|<rdkit.Chem.rdche...|       30|            1.8592|          60|
|CC1CC(C[SH](=O)(O...| 10.7|<rdkit.Chem.rdche...|       29|1.8693999999999995|          58|
|CS(=O)(=O)N1CCC(O...| 10.7|<rdkit.Chem.rdche...|       27|0.8265999999999994|  

In [ ]:
spark_df.write.parquet(
    "parquet_df",
    mode="overwrite"
)

In [ ]:
final_spark_df = spark.read.parquet("parquet_df")
final_spark_df.show(10)

+--------------------+-----+--------------------+---------+------------------+------------+
|              SMILES|pIC50|                 mol|num_atoms|              logP|double_atoms|
+--------------------+-----+--------------------+---------+------------------+------------+
|O=S(=O)(Nc1cccc(-...| 4.26|<rdkit.Chem.rdche...|       25| 4.159100000000002|          50|
|O=c1cc(-c2nc(-c3c...| 4.34|<rdkit.Chem.rdche...|       36|3.6743000000000015|          72|
|NC(=O)c1ccc2c(c1)...| 4.53|<rdkit.Chem.rdche...|       23|            1.5361|          46|
|NCCCn1c(C2CCNCC2)...| 4.56|<rdkit.Chem.rdche...|       22|0.9509999999999998|          44|
|CNC(=S)Nc1cccc(-c...| 4.59|<rdkit.Chem.rdche...|       21| 3.213000000000001|          42|
|O=C(Nc1cccc(-c2cn...|  4.6|<rdkit.Chem.rdche...|       22|3.6453000000000015|          44|
|O=C(Nc1cccc(-c2cn...| 4.61|<rdkit.Chem.rdche...|       24| 4.142100000000002|          48|
|COc1cc(Nc2nccc(-c...| 4.67|<rdkit.Chem.rdche...|       26| 4.566400000000004|  

In [112]:
raw_file = "/content/SMILES_Big_Data_Set.csv"

In [127]:
def clean_data():
  data = pd.read_csv(raw_file)
  data["pIC50"] = data["pIC50"].fillna(data["pIC50"].mean())
  data = data[data["num_atoms"]<60].copy()
  data.to_csv("clean_data.csv",index=False)


In [128]:
def validate_smiles():
  data = pd.read_csv("clean_data.csv")
  valid_rows = []
  valid_count = 0
  invalid_count = 0
  for i, smile in enumerate (data["SMILES"]):
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
      invalid_count += 1
    else:
      valid_count += 1
      valid_rows.append(data.iloc[i])

  df_valid_rows = pd.DataFrame(valid_rows)
  print(f"Valid SMILES: {valid_count}, Invalid SMILES: {invalid_count}")
  df_valid_rows.to_csv("valid_smiles.csv",index=False)

In [129]:
def generate_descriptors():
  data = pd.read_csv("valid_smiles.csv")
  weight=[]
  atoms=[]
  bonds=[]
  tpsa=[]
  numhdoner=[]
  numhaccept=[]

  for i in data["SMILES"]:
    mol = Chem.MolFromSmiles(i)
    weight_value = Descriptors.MolWt(mol)
    atoms_value = mol.GetNumAtoms()
    bonds_value = mol.GetNumBonds()
    tpsa_value = Descriptors.TPSA(mol)
    numhdoner_value = Descriptors.NumHDonors(mol)
    numhaccept_value = Descriptors.NumHAcceptors(mol)
    weight.append(weight_value)
    atoms.append(atoms_value)
    bonds.append(bonds_value)
    tpsa.append(tpsa_value)
    numhdoner.append(numhdoner_value)
    numhaccept.append(numhaccept_value)
  data["Molecular_Weight"] = weight
  data["Num_Atoms"] = atoms
  data["Num_Bonds"] = bonds
  data["TPSA"] = tpsa
  data["NumHDoner"] = numhdoner
  data["NumHAcceptor"] = numhaccept
  data.to_csv("Descriptors_values.csv",index=False)

In [130]:
def train_and_evaluate():
  data = pd.read_csv("Descriptors_values.csv")
  x = data.drop(["SMILES","pIC50","mol"],axis=1)
  y = data["pIC50"]
  x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=20
  )
  models = {
    "linear_regression :" : LinearRegression(),
    "decision_tree_regressor :" : DecisionTreeRegressor(),
    "random_forest_regressor :" : RandomForestRegressor(
        max_depth=15,
        n_estimators=500,
        min_samples_split=5,
        min_samples_leaf=5,
        n_jobs=-1
    )
  }
  for name, model in models.items():
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    print(name)
    print("mean_absolute_error :",mean_absolute_error(y_test,y_pred))
    print("mean_squared_error :",mean_squared_error(y_test,y_pred))
    print("r2_score :",r2_score(y_test,y_pred))

In [131]:
with DAG(
    dag_id = "cadd_data_pipeline",
    start_date = datetime(2026,5,8),
    schedule="@daily",
    catchup=False
) as dag:

  task1 = PythonOperator(
      task_id = "clean_data",
      python_callable=clean_data
  )

  task2 = PythonOperator(
      task_id="validate_smiles",
      python_callable=validate_smiles
  )

  task3 = PythonOperator(
      task_id="generate_descriptors",
      python_callable=generate_descriptors
  )

  task4 = PythonOperator(
      task_id="train_and_evaluate",
      python_callable=train_and_evaluate
  )

  task1 >> task2 >> task3 >> task4

In [132]:
clean_data()


In [133]:
validate_smiles()

Valid SMILES: 16067, Invalid SMILES: 0


In [134]:
generate_descriptors()

In [135]:
train_and_evaluate()

linear_regression :
mean_absolute_error : 1.3262702166121252
mean_squared_error : 3.5886079199878176
r2_score : 0.3590541164915093
decision_tree_regressor :
mean_absolute_error : 0.3702536006358209
mean_squared_error : 2.223891709924152
r2_score : 0.6028002310017243
random_forest_regressor :
mean_absolute_error : 0.41590459224775933
mean_squared_error : 1.3560152044821279
r2_score : 0.7578079348131477
